In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/retail_store_inventory.csv")

df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df["Store ID"].unique() # What Stores exist?

In [ ]:
df["Product ID"].unique() # What products exist?

In [ ]:
df["Category"].unique() # What categories exist?

In [ ]:
df["Region"].unique() # What are the different regions?

In [ ]:
df["Date"].min(), df["Date"].max() # What is the date range?

In [ ]:
df.isna().sum() # Any missing values?

In [ ]:
df.duplicated().sum() # Any duplicate rows?

In [ ]:
# Verify the number of daily observations per Store × Product combination

# and how many combinations have each observation count.

# Output: 731 observations     100 Store × Product combinations

df.groupby(["Store ID", "Product ID"]).size().value_counts().sort_index()

In [ ]:
# Verify the first few dates represented in the dataset

df["Date"].value_counts().sort_index().head()

In [ ]:
# Verify the last few dates represented in the dataset

df["Date"].value_counts().sort_index().tail()

In [ ]:
# Verify that every date has the expected number of Store × Product observations

df.groupby("Date").size().value_counts().sort_index()

In [ ]:
# Check the number of unique values in every column

df.nunique()

In [ ]:
# Check the data type assigned to each column to ensure Pandas interpreted the dataset correctly

df.dtypes

In [ ]:
# Convert the Date column from text (object) into Pandas' datetime type for proper date-based analysis

df["Date"] = pd.to_datetime(df["Date"])

In [ ]:
# Verify that the Date column was successfully converted to datetime

df["Date"].dtype

In [ ]:
# Confirm that the dataset still has 731 unique dates after converting Date to datetime

df["Date"].nunique()

In [ ]:
# Verify that each Store × Product combination has one observation for every date in the dataset

store_product_date_counts = df.groupby(["Store ID", "Product ID"])["Date"].nunique()

store_product_date_counts.value_counts().sort_index()

In [ ]:
# Display all columns and their Pandas data types as a quick data dictionary reference

data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

data_dictionary

In [ ]:
# Summarize the minimum, maximum, mean, and median of each numerical column to identify suspicious values

df.describe().T[["min", "max", "mean", "50%"]]

### Dataset Quality & Structure

The dataset is clean and structurally consistent, with 73,100 rows across 5 stores, 20 products, and 731 days. 

No missing values or duplicates were found. 

Most numerical ranges look reasonable, but negative demand forecasts require further investigation.

In [ ]:
# Find all rows where the forecasted demand is negative so we can investigate the anomaly

negative_forecasts = df[df["Demand Forecast"] < 0]

negative_forecasts

In [ ]:
# Count how many negative demand forecasts exist and what percentage of the dataset they represent
print("Negative forecasts:", len(negative_forecasts))
print("Percentage of dataset:", len(negative_forecasts) / len(df) * 100)

### Anomaly Investigation Logic

If negative forecasts are spread fairly evenly across the dataset, they are likely generated-data noise. If they are heavily concentrated in a specific product, store, or season, that may indicate a meaningful pattern worth investigating.

In [ ]:
# Count negative demand forecasts for each product to see whether the anomaly is concentrated in specific products

negative_forecasts["Product ID"].value_counts()

In [ ]:
# Count negative demand forecasts for each store to see whether the anomaly is concentrated in specific stores

negative_forecasts["Store ID"].value_counts()

In [ ]:
# Count negative demand forecasts by season to see whether the anomaly is related to seasonal patterns

negative_forecasts["Seasonality"].value_counts()

### Anomaly Investigation Conclusion

The negative forecasts appear to be distributed across the dataset rather than concentrated in one product, store, or season.

This suggests synthetic-data noise or anomaly generation rather than a meaningful business pattern.

I will document the finding, retain the values for now, and move on.

In [ ]:
# Measure the linear relationship between inventory and sales to see whether higher inventory generally corresponds to higher sales

df[["Inventory Level", "Units Sold"]].corr()

For reference, 

The correlation ranges from -1 to +1:

+1 → strong positive relationship

0 → little/no linear relationship

-1 → strong negative relationship

In [ ]:
# Calculate the average absolute difference between forecasted demand and actual units sold

# to get an initial measure of how far the forecasts are from observed sales.

forecast_error = (df["Demand Forecast"] - df["Units Sold"]).abs().mean()

print("Mean Absolute Forecast Error:", forecast_error)

An MAE of 8.34 establishes that the forecasts are reasonably close to actual sales on average.

In [ ]:
# Calculate forecast error for every row so we can distinguish over-forecasting from under-forecasting

df["Forecast Error"] = df["Demand Forecast"] - df["Units Sold"]

# Display summary statistics of the forecast error

df["Forecast Error"].describe()

The forecast error has a mean of +5.03 units, indicating a consistent tendency to over-forecast demand by approximately 5 units. 

The similar median error of +4.99 suggests that this bias is widespread rather than driven only by extreme observations.

In [ ]:
# Measure how strongly forecasted demand and actual sales move together

df[["Demand Forecast", "Units Sold"]].corr()

### Forecast vs Actual Demand

Forecast and Units Sold have an extremely strong positive correlation of 0.997, indicating that the forecasts closely track actual demand. 

Combined with the +5.03 average forecast error, the dataset shows highly aligned forecasts with a slight over-forecasting bias.

In [ ]:
# Calculate key inventory percentiles to understand what "low" and "high" inventory look like in this dataset

df["Inventory Level"].quantile([0.10, 0.25, 0.50, 0.75, 0.90])

In [ ]:
# Define the 10th percentile as a temporary threshold for identifying unusually low inventory

low_inventory_threshold = df["Inventory Level"].quantile(0.10)

low_inventory_threshold

In [ ]:
# Count how many observations fall into the bottom 10% of inventory levels

low_inventory_count = (df["Inventory Level"] <= low_inventory_threshold).sum()

print("Low inventory observations:", low_inventory_count)

print("Percentage of dataset:", low_inventory_count / len(df) * 100)

In [ ]:
# Count low-inventory observations for each store to identify stores that may experience inventory pressure more frequently

df[df["Inventory Level"] <= low_inventory_threshold]["Store ID"].value_counts()

In [ ]:
# Calculate the percentage of observations with low inventory for each store
# so stores can be compared using a normalized metric rather than raw counts.

low_inventory_by_store = (
    df.assign(Low_Inventory=df["Inventory Level"] <= low_inventory_threshold)
      .groupby("Store ID")["Low_Inventory"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

low_inventory_by_store

### Store-Level Inventory Analysis
Low-inventory observations are relatively evenly distributed across stores, ranging from 9.58% to 10.68%. S002 has the highest rate, while S003 has the lowest, but the small difference suggests no significant store-level imbalance.

In [ ]:
# Calculate the percentage of observations with low inventory for each product
# to identify products that experience inventory pressure more frequently.

low_inventory_by_product = (
    df.assign(Low_Inventory=df["Inventory Level"] <= low_inventory_threshold)

      .groupby("Product ID")["Low_Inventory"]

      .mean()

      .mul(100)

      .sort_values(ascending=False)
)

low_inventory_by_product

### Product-Level Inventory Analysis
Low-inventory observations are relatively evenly distributed across products, ranging from 9.29% to 10.97%. P0006 has the highest rate and P0014 the lowest, but the small difference suggests no significant product-level imbalance.

In [ ]:
# Calculate an approximate inventory coverage ratio to estimate how many days
# the current inventory could cover at the observed daily sales rate.

df["Inventory Coverage"] = df["Inventory Level"] / df["Units Sold"].replace(0, np.nan) # Replace zero sales values with NaN so they don't create invalid division results

df["Inventory Coverage"].describe()

In [ ]:
# Summarize the distribution of inventory coverage to understand typical and extreme coverage levels.

df["Inventory Coverage"].quantile([0.10, 0.25, 0.50, 0.75, 0.90])

In [ ]:
# Calculate inventory coverage using forecasted demand as the expected daily demand rate
# so coverage reflects anticipated demand rather than only today's sales.

df["Forecast Coverage"] = (
    df["Inventory Level"] / df["Demand Forecast"].replace(0, np.nan)
)

In [ ]:
# Examine the distribution of forecast-based inventory coverage to understand expected stock duration

df["Forecast Coverage"].quantile([0.10, 0.25, 0.50, 0.75, 0.90])

### Forecast-Based Inventory Coverage

The median inventory coverage is about 1.89 days.

25% of the observations have less than 1.28 days of coverage, while 75% have less than 3.57 days.

This means a large portion of the data has relatively low inventory coverage based on forecasted demand.

In [ ]:
# Identify observations where available inventory is less than one day of forecasted demand

low_coverage = df["Forecast Coverage"] < 1

print("Low coverage observations:", low_coverage.sum())

print("Percentage of dataset:", low_coverage.mean() * 100)

### Low Inventory Coverage

3,258 observations (4.46%) have less than 1 day of inventory coverage based on forecasted demand.

These are potential inventory-risk situations that we can investigate later with ShelfSleuth.

-----------------------------------------------------------------------------------------------------------------------------------------------------

### Move Data to DuckDB

We have finished the basic data exploration.

Now we'll put our DataFrame into a SQL database so we can start working with SQL and eventually build our Text-to-SQL and semantic layer components.

In [ ]:
# Import DuckDB and Path so we can create and connect to our local database

import duckdb

from pathlib import Path


# Create the folder where our DuckDB database will be stored

db_dir = Path("../data/processed")

db_dir.mkdir(parents=True, exist_ok=True)


# Create or open the ShelfSleuth DuckDB database

db_path = db_dir / "shelfsleuth.duckdb"

con = duckdb.connect(str(db_path))

In [ ]:
# Register our Pandas DataFrame with DuckDB so DuckDB can access it

con.register("retail_df", df)


# Copy the DataFrame into a persistent SQL table called retail_inventory

con.execute("""
    CREATE OR REPLACE TABLE retail_inventory AS
    SELECT * FROM retail_df
""")

In [ ]:
# Check that our SQL table contains all 73,100 rows from the original DataFrame

con.execute("SELECT COUNT(*) FROM retail_inventory").fetchone()

### SQL Exploration

Our data is now stored as a SQL table in DuckDB.

We'll use a few SQL queries to understand how the table works before building the semantic layer on top of it.

In [ ]:
# Show the SQL schema so we can see the columns and data types available to the Text-to-SQL agent

con.execute("DESCRIBE retail_inventory").fetchdf()

In [ ]:
# Find the average inventory level for each store using SQL

con.execute("""
    SELECT
        "Store ID",
        AVG("Inventory Level") AS avg_inventory

    FROM retail_inventory

    GROUP BY "Store ID"

    ORDER BY avg_inventory DESC
""").fetchdf()

### Semantic Layer

The database tells us what the columns are, but not what they mean from a business perspective.

The semantic layer will define things like metrics, dimensions, and business terms so the AI knows how to translate user questions into the right SQL.

In [ ]:
# Add the ShelfSleuth project root to Python's import path so the notebook can access our project modules

import sys

from pathlib import Path


# Move one level up from the notebooks directory to reach the ShelfSleuth project root

project_root = Path.cwd().parent # cwd --> current working directory

sys.path.append(str(project_root))

In [ ]:
# Import the semantic layer so we can inspect the business definitions available to the AI.

from semantic_layer.metrics import get_semantic_context # get_semantic_context is a function in ShelfSleuth//semantic_layer/metrics.py


# Retrieve the complete semantic context.

semantic_context = get_semantic_context()


# Display the semantic definitions.

semantic_context

In [ ]:
# Import the Text-to-SQL agent so we can connect it to DuckDB and our semantic layer

from agents.text_to_sql_agent import TextToSQLAgent


# Create the Text-to-SQL agent

text_to_sql_agent = TextToSQLAgent(
    database_connection=con,

    semantic_context=semantic_context,
)


# Display the agent to confirm it was created successfully

text_to_sql_agent

In [ ]:
# Test whether our Text-to-SQL agent can execute a SQL query against DuckDB

test_sql = """
SELECT
    "Store ID",
    AVG("Inventory Level") AS avg_inventory
FROM retail_inventory
GROUP BY "Store ID"
ORDER BY avg_inventory DESC
"""


# Execute the SQL through our agent

result = text_to_sql_agent.execute_sql(test_sql)


# Display the result

result

In [ ]:
# Ask the Text-to-SQL agent to convert a natural-language question into SQL

question = "Which store has the lowest average inventory?"


# Generate SQL using Gemini

generated_sql = text_to_sql_agent.generate_sql(question)


# Display the SQL generated by Gemini

print(generated_sql)

In [ ]:
# Execute the SQL generated by Gemini against our DuckDB database

result = text_to_sql_agent.execute_sql(generated_sql)


# Display the final result returned by the database

result

In [ ]:
# Ask the agent to answer a business question using its complete SQL generation and execution loop

question = "Which store has the lowest average inventory?"


# Let the agent generate SQL, execute it, and automatically correct SQL errors if necessary

answer = text_to_sql_agent.answer_question(question)


# Display the SQL generated by the agent

print(answer["sql"])


# Display the final database result

answer["result"]

### Text-to-SQL Agent Loop

The agent now does the same process as our previous generate_sql() → execute_sql() snippets, but handles everything through one answer_question() method.

It can also detect SQL errors, ask Gemini to correct them, and retry.

This is our first real agent loop.

Next: OKF/Knowledge Agent — combining SQL results with business knowledge to explain why something is happening.

In [ ]:
from agents.knowledge_agent import KnowledgeAgent
from okf.loader import load_okf_knowledge


# Load business knowledge from the OKF Markdown bundle
knowledge_base = load_okf_knowledge()


# Create the Knowledge Agent
knowledge_agent = KnowledgeAgent(
    knowledge_base=knowledge_base
)

In [ ]:
from tools.investigation_tool import InvestigationTool

investigation_tool = InvestigationTool(
    database_connection=con
)

print("Investigation tool initialized successfully.")

In [ ]:
import importlib

import agents.action_planner_agent
import agents.critic_agent
import agents.research_pipeline

importlib.reload(agents.action_planner_agent)
importlib.reload(agents.critic_agent)
importlib.reload(agents.research_pipeline)

print("ShelfSleuth agents reloaded successfully.")

In [ ]:
from agents.research_pipeline import ShelfSleuthPipeline

pipeline = ShelfSleuthPipeline(
    database_connection=con,
    semantic_context=semantic_context,
    knowledge_base=KNOWLEDGE_BASE,
    investigation_tool=investigation_tool,
)

print("ShelfSleuth pipeline initialized successfully.")

In [ ]:
answer = pipeline.run(question)

print("=" * 80)
print("SHELF SLEUTH INVESTIGATION")
print("=" * 80)

print("\nQUESTION")
print(answer["question"])

print("\nSQL")
print(answer["sql"])

print("\nSQL RESULT")
print(answer["sql_result"])

print("\nBUSINESS KNOWLEDGE")
print(answer["knowledge"])

print("\nROOT CAUSE ANALYSIS")
print(answer["root_cause_analysis"])

print("\nACTION PLAN")
print(answer["action_plan"])

print("\nCRITIC REVIEW")
print(answer["critic_review"])

### **Robustness & Evaluation**

We validated the ShelfSleuth pipeline using deterministic DuckDB queries and a small set of business questions.

The end-to-end pipeline was successfully executed on a representative investigation question, and the generated SQL result matched the deterministic database result.

Additional live evaluations were attempted, but the Gemini API free-tier request quota was exhausted during testing. These cases are therefore treated as **unevaluated due to API limits**, rather than failures of the ShelfSleuth pipeline.

For a larger evaluation, the same test cases can be executed using additional API capacity or a mocked/cached LLM layer.

------------------------------------------------------------------------------------------------------------------------------------------------------
#### **Conclusion**

 ShelfSleuth demonstrates an end-to-end AI investigation workflow that combines semantic context, business knowledge, Text-to-SQL, root-cause analysis, action planning, and automated critique to turn retail data questions into actionable business insights.

------------------------------------------------------------------------------------------------------------------------------------------------------